# CLV 이중축 체크포인트 진단 (두 데이터셋 연속 실행)

기존 `dual_clv_fixed` 체크포인트를 다시 학습하지 않고 N축만, V축만, N+V를 validation에서 재평가합니다. Dunnhumby 전체와 H&M 60일을 순서대로 처리하며 test/holdout은 열지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'TO_BE_PINNED'
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
assert __import__('subprocess').check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA


In [ ]:
from pathlib import Path
from IPython.display import display
import pandas as pd
import torch
from lightgcn_clv_dual_checkpoint_diagnostic import run_checkpoint_diagnostic

print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
if not torch.cuda.is_available():
    print('경고: CPU에서도 실행되지만 전체 상품 점수 계산이 오래 걸릴 수 있습니다.')

DRIVE_ROOT = Path('/content/drive/MyDrive/논문/data')
RUNS = [
    ('dunnhumby', DRIVE_ROOT / 'results_clv_dual_dunnhumby'),
    ('hm_w60', DRIVE_ROOT / 'results_clv_dual_hm_w60'),
]

def latest_screening_json(folder):
    candidates = list(folder.glob('clv_dual_*.json'))
    if not candidates:
        raise FileNotFoundError(f'원본 dual 결과 JSON이 없습니다: {folder}')
    return max(candidates, key=lambda path: path.stat().st_mtime)

for label, folder in RUNS:
    print(label, '원본:', latest_screening_json(folder))


## 실행

이 셀은 학습 함수를 호출하지 않습니다. 원자료에서 동일 validation split만 재구성하고 저장된 모델의 점수를 다시 계산합니다.

In [ ]:
results = {}
for label, folder in RUNS:
    source_json = latest_screening_json(folder)
    output_dir = folder / 'checkpoint_diagnostics'
    print(f'\n===== {label} 진단 시작 =====')
    results[label] = run_checkpoint_diagnostic(source_json, output_dir)
    print(f'===== {label} 완료 =====')
    print(results[label].attrs['result_paths'])


In [ ]:
for label, frame in results.items():
    spec = frame.attrs['diagnostic_spec']
    selected = 2.0 if label == 'dunnhumby' else 1.0
    print(f'\n===== {label}: 기존 선택 lambda={selected} 축별 비교 =====')
    columns = ['model_id', 'gate_shape', 'lambda', 'effective_strength',
               'recall@10', 'ndcg@10', 'revenue@10', 'arp@10',
               'coverage@10', 'n_distinct@10', 'eff_catalog@10',
               'top10_share@10', 'top100_share@10']
    chosen = frame[frame['model_id'].eq('m1') | frame['lambda'].eq(selected)]
    display(chosen[columns].sort_values(['model_id', 'lambda']))
    q = pd.read_csv(frame.attrs['result_paths']['quadrant_csv'])
    display(q[q['lambda'].eq(selected) & q['metric'].eq('revenue')][
        ['model_id', 'quadrant', 'user_count', 'baseline_mean', 'model_mean',
         'mean_delta', 'lo', 'hi', 'improved_user_share']
    ].sort_values(['model_id', 'quadrant']))

print('완료. 각 결과 폴더의 checkpoint_diagnostics 아래 CSV/JSON/PNG를 공유해 주세요.')
